# PMC oa_comm Embedding Pipeline

使用 `BAAI/bge-base-en-v1.5` 对切分好的 chunk 批次目录进行向量化，写入 ChromaDB 持久化索引。支持断点续传。

## 0. 配置

In [ ]:
import os, sys
os.environ.setdefault('OMP_NUM_THREADS', str(os.cpu_count() or 4))
os.environ['PYTHONUTF8'] = '1'
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')
# 解决显存碎片导致的 OOM（必须在 import torch 之前设置）
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

from pathlib import Path

# ── 修改这里 ─────────────────────────────────────────────────────
BATCH_DIR   = Path('/root/autodl-tmp/pipeline_output/batches_full')
DB_DIR      = Path('/root/autodl-tmp/pipeline_output/chroma_db')
COLLECTION  = 'pmc_full'
MODEL       = 'BAAI/bge-base-en-v1.5'
EMBED_BATCH = 128    # OOM 时继续往下调（64）
CHROMA_BATCH= 2000
RESUME      = True
OVERWRITE   = False
# ─────────────────────────────────────────────────────────────────

batch_files = sorted(BATCH_DIR.glob('batch_*.parquet'))
print(f'批次目录  : {BATCH_DIR}')
print(f'批次文件数: {len(batch_files):,}')
print(f'ChromaDB  : {DB_DIR}  集合={COLLECTION}')
print(f'HF镜像    : {os.environ["HF_ENDPOINT"]}')

## 1. 加载模型与索引

In [ ]:
import torch, os
sys.path.insert(0, str(Path('~/autodl-tmp').expanduser()))  # AutoDL 路径
from pmc_vector_index import BGEEmbedder, PMCVectorIndex, setup_logging

# AutoDL 无法直连 HuggingFace，优先用镜像或本地缓存
# 方式1：镜像（推荐，大陆服务器）
os.environ.setdefault('HF_ENDPOINT', 'https://hf-mirror.com')

# 方式2：如果已下载到本地，直接填本地路径
# MODEL = '/root/autodl-tmp/models/bge-base-en-v1.5'  # 取消注释并填实际路径

DB_DIR.mkdir(parents=True, exist_ok=True)
log_path = DB_DIR / f'embed_{COLLECTION}.log'
log = setup_logging(log_path)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU   : {torch.cuda.get_device_name(0)}')
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

embedder = BGEEmbedder(
    model_name = MODEL,
    batch_size = EMBED_BATCH,
    device     = device,
    log        = log,
)

index = PMCVectorIndex(
    db_dir          = DB_DIR,
    collection_name = COLLECTION,
    embedder        = embedder,
    log             = log,
)

print(f'\n当前集合已有向量: {index.collection.count():,}')

## 2. 查看断点进度（可选）

In [ ]:
import json

ckpt_path = BATCH_DIR / f'embed_checkpoint_{COLLECTION}.json'
if ckpt_path.exists():
    with open(ckpt_path) as f:
        ckpt = json.load(f)
    done = set(ckpt.get('done_files', []))
    total = len(batch_files)
    pending = total - len(done)
    print(f'断点文件  : {ckpt_path.name}')
    print(f'已完成批次: {len(done):,} / {total:,}  ({len(done)/total*100:.1f}%)')
    print(f'待处理批次: {pending:,}')
    
    # 估算剩余时间（基于已有向量数推算速度）
    n_vec = index.collection.count()
    if len(done) > 0 and n_vec > 0:
        avg_per_batch = n_vec / len(done)
        remaining_chunks = pending * avg_per_batch
        print(f'已有向量  : {n_vec:,}')
        print(f'平均每批  : {avg_per_batch:,.0f} chunks')
        print(f'估算剩余  : {remaining_chunks:,.0f} chunks')
else:
    print('无断点文件，将从头开始')

## 3. 运行 Embedding

> 这个 cell 会持续运行直到所有批次完成。进度输出在日志文件和下方。

In [ ]:
import time

t_start = time.time()
print(f'开始时间: {time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'日志文件: {log_path}')
print('─' * 60)

stats = index.build_from_dir(
    batch_dir    = BATCH_DIR,
    chroma_batch = CHROMA_BATCH,
    overwrite    = OVERWRITE,
    resume       = RESUME,
)

elapsed = time.time() - t_start
print('─' * 60)
print(f'完成时间  : {time.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'总耗时    : {elapsed/3600:.2f} 小时')
print(f'本次新增  : {stats["new_chunks_this_run"]:,}')
print(f'集合总量  : {stats["total_chunks"]:,}')

## 4. 保存统计信息

In [ ]:
import json

stats_path = DB_DIR / f'index_stats_{COLLECTION}.json'
index.save_stats(stats, stats_path)
print(f'统计已保存: {stats_path}')

print('\n── 统计摘要 ─────────────────────────────────────────────')
for k, v in stats.items():
    if k != 'metadata_fields':
        print(f'  {k:<25}: {v}')

## 5. 验证索引（快速抽样）

In [ ]:
# 随机抽取几个 query 测试检索效果
test_queries = [
    'CRISPR gene editing therapy',
    'COVID-19 vaccine efficacy clinical trial',
    'machine learning drug discovery',
]

for q in test_queries:
    result = index.query(q, n_results=3)
    print(f'\nQuery: {q}')
    for r in result['results']:
        print(f'  [{r["rank"]}] sim={r["similarity"]:.4f}  '
              f'type={r["metadata"].get("chunk_type","")}  '
              f'imrad={r["metadata"].get("imrad_type","")}  '
              f'journal={r["metadata"].get("journal","")[:30]}')
        print(f'       {r["text_preview"][:100]}')

## 6. 元数据过滤查询（可选）

In [ ]:
# 只在 methods 段落中检索，且限定 2020 年以后的文章
result = index.query(
    query_text   = 'RNA sequencing single cell analysis protocol',
    n_results    = 5,
    where_filter = {
        '$and': [
            {'imrad_type': {'$eq': 'methods'}},
            {'pub_year':   {'$gte': 2020}},
        ]
    }
)

print(f'Query: {result["query"]}')
print(f'过滤: methods + pub_year >= 2020\n')
for r in result['results']:
    meta = r['metadata']
    print(f'[{r["rank"]}] sim={r["similarity"]:.4f}  year={meta.get("pub_year")}  '
          f'journal={meta.get("journal","")[:35]}')
    print(f'     section: {meta.get("section_title","")}')
    print(f'     {r["text_preview"][:120]}')
    print()